# Part 2: Processing Stages & Supply Chain Edges

Sections 3A–3E: Processing stage classification, smelter matching, product form assignment, downstream edges, unified edge table.

**Depends on:** Part 1


In [1]:
import os, shutil, re, pickle
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

SALIDAS_PATH = os.path.join(BASE_DIR, "data", "salidas_2024_clean.csv")

# ── Load state from Part 2 ───────────────────────────────────────────
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_2.pkl")
with open(_state_path, "rb") as _f:
    _state = pickle.load(_f)

inv = _state["inv"]
links = _state["links"]
comm_col = _state.get("comm_col", "COMMODITY_LIST_STR")
idle_mines = _state.get("idle_mines", set())
COMPANY_TO_DEPOSIT = _state.get("COMPANY_TO_DEPOSIT", {})
CODELCO_EXTRA_SEARCH = _state.get("CODELCO_EXTRA_SEARCH", {})
SMELTERS = _state.get("SMELTERS", [])
PORTS = _state.get("PORTS", [])
SMELTER_NAME_MAP = _state.get("SMELTER_NAME_MAP", {})
DEDICATED_PORT = _state.get("DEDICATED_PORT", {})
CODELCO_CATHODE_ROUTING = _state.get("CODELCO_CATHODE_ROUTING", {})

print(f"Loaded state from Part 2: {len(inv)} inv rows, {len(links)} link rows")

# ── Shared utility functions ──────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return list(matched)


Loaded state from Part 2: 461 inv rows, 1109 link rows


In [2]:

# ── 3A. Processing stage classification (vectorized) ─────────────────────

section_header("3A. PROCESSING STAGE CLASSIFICATION")

STAGE_MAP = {
    "Mine (active)": "extraction", "Mine (idle)": "extraction_idle",
    "Mine (USGS)": "extraction", "Prospect/Project": "extraction",
    "Concentrator": "concentration", "SX-EW Plant": "sx_ew",
    "Smelter": "smelting", "Refinery": "refining",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}
inv["CHAIN_STAGE"] = inv["FACILITY_TYPE"].map(STAGE_MAP).fillna("other")

# Refine "processing" based on facility name keywords (vectorized)
proc_mask = inv["CHAIN_STAGE"] == "processing"
name_lower = inv["FACILITY_NAME"].str.lower()
inv.loc[proc_mask & name_lower.str.contains("smelter|fundici|smelting", na=False), "CHAIN_STAGE"] = "smelting"
inv.loc[proc_mask & name_lower.str.contains("refin|electro", na=False) & (inv["CHAIN_STAGE"] == "processing"), "CHAIN_STAGE"] = "refining"
inv.loc[proc_mask & name_lower.str.contains("sx-ew|sx/ew|leach|lixiv|cathode|catod", na=False) & (inv["CHAIN_STAGE"] == "processing"), "CHAIN_STAGE"] = "sx_ew"
inv.loc[proc_mask & name_lower.str.contains("concentrat|flotation|mill", na=False) & (inv["CHAIN_STAGE"] == "processing"), "CHAIN_STAGE"] = "concentration"

for stage, count in inv["CHAIN_STAGE"].value_counts().items():
    print(f"  {stage:<18} {count:>4}")

# ── 3B. Smelter matching to inventory ────────────────────────────────────

section_header("3B. SMELTER AND PORT SETUP")

smelter_inv_map = {}
for sm in SMELTERS:
    for term in sm["search"]:
        hits = inv[inv["FACILITY_NAME"].str.contains(term, case=False, na=False, regex=False)]
        if len(hits) > 0:
            idx = hits.index[0]
            smelter_inv_map[sm["name"]] = inv.at[idx, "FACILITY_NAME"]
            inv.at[idx, "CHAIN_STAGE"] = "smelting"
            print(f"  {sm['name']:<30} -> {inv.at[idx, 'FACILITY_NAME']}")
            break
    else:
        print(f"  {sm['name']:<30} -> NOT IN INVENTORY")

ports_df = pd.DataFrame(PORTS)
ports_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Ports.csv"), index=False)
print(f"\n  Ports saved: {len(ports_df)}")

# ── 3C. Product form assignment (vectorized) ─────────────────────────────

section_header("3C. PRODUCT FORM + DOWNSTREAM EDGES")

pt = links.get("PLANT_TYPE", pd.Series("", index=links.index)).str.lower()
pn = links.get("PLANT_NAME", pd.Series("", index=links.index)).str.lower()

conditions = [
    pt.str.contains("sx-ew|sx/ew", na=False) | pn.str.contains("sx-ew|sx/ew|leach|cathode", na=False),
    pt.str.contains("smelter", na=False) | pn.str.contains("smelter|fundici", na=False),
    pt.str.contains("refin", na=False) | pn.str.contains("refin|electro", na=False),
    pt.str.contains("concentrat", na=False) | pn.str.contains("concentrat|flotation|mill", na=False),
]
choices = ["cathode_sxew", "blister", "cathode_er", "concentrate"]
links["PRODUCT_FORM"] = np.select(conditions, choices, default="concentrate")

# Log defaults
n_defaulted = (~np.any(conditions, axis=0)).sum()
print("Product forms in mine-plant links:")
print(links["PRODUCT_FORM"].value_counts().to_string())
if n_defaulted > 0:
    print(f"\n  Note: {n_defaulted} links defaulted to 'concentrate' (no keyword match)")

# ── 3D. Build downstream edges ───────────────────────────────────────────

downstream_edges = []
concentrators = inv[inv["CHAIN_STAGE"] == "concentration"].copy()
smelter_fed = set()

# A. Concentrator -> Smelter (named feed mines + regional feed)
for sm in SMELTERS:
    sm_inv_name = smelter_inv_map.get(sm["name"], sm["name"])

    # Named feeds
    for mine_term in sm.get("feeds_from_mines", []):
        concs = concentrators[concentrators["FACILITY_NAME"].str.contains(
            mine_term, case=False, na=False, regex=False)]
        for cidx, crow in concs.iterrows():
            downstream_edges.append({
                "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
                "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
                "TO_NAME": sm_inv_name, "TO_TYPE": "smelter",
                "TO_LAT": sm["lat"], "TO_LON": sm["lon"],
                "EDGE_TYPE": "concentrate_to_smelter", "PRODUCT_FORM": "concentrate",
                "OPERATOR": sm["operator"],
                "DISTANCE_KM": haversine_km(crow["LATITUD"], crow["LONGITUD"], sm["lat"], sm["lon"])
                if pd.notna(crow["LATITUD"]) else None,
            })
            smelter_fed.add(cidx)

    # Regional feed for custom smelters (Altonorte, Paipote)
    if sm.get("feeds_from_region") and not sm.get("feeds_from_mines"):
        region = sm["feeds_from_region"]
        regional_concs = concentrators[
            concentrators.get("REGION", pd.Series("", index=concentrators.index)).str.contains(region, case=False, na=False) &
            ~concentrators.index.isin(smelter_fed)
        ]
        for cidx, crow in regional_concs.iterrows():
            if pd.isna(crow["LATITUD"]): continue
            dist = haversine_km(crow["LATITUD"], crow["LONGITUD"], sm["lat"], sm["lon"])
            if dist <= 300:
                downstream_edges.append({
                    "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
                    "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
                    "TO_NAME": sm_inv_name, "TO_TYPE": "smelter",
                    "TO_LAT": sm["lat"], "TO_LON": sm["lon"],
                    "EDGE_TYPE": "concentrate_to_smelter", "PRODUCT_FORM": "concentrate",
                    "OPERATOR": sm["operator"], "DISTANCE_KM": round(dist, 1),
                })
                smelter_fed.add(cidx)

# B. Smelter -> Port
for sm in SMELTERS:
    sm_inv_name = smelter_inv_map.get(sm["name"], sm["name"])
    for port_name in sm["export_ports"]:
        port = next((p for p in PORTS if p["name"] == port_name), None)
        if port:
            downstream_edges.append({
                "FROM_NAME": sm_inv_name, "FROM_TYPE": "smelter",
                "FROM_LAT": sm["lat"], "FROM_LON": sm["lon"],
                "TO_NAME": port["name"], "TO_TYPE": "port",
                "TO_LAT": port["lat"], "TO_LON": port["lon"],
                "EDGE_TYPE": "smelter_to_port", "PRODUCT_FORM": sm["output_product"],
                "OPERATOR": sm["operator"],
                "DISTANCE_KM": haversine_km(sm["lat"], sm["lon"], port["lat"], port["lon"]),
            })

# C. Concentrator -> Port (dedicated overrides first, then nearest)
for cidx, crow in concentrators.iterrows():
    if pd.isna(crow["LATITUD"]) or cidx in smelter_fed:
        continue
    # Check for dedicated port override
    facility_name = str(crow["FACILITY_NAME"]).lower()
    assigned_port = None
    for key, port_name in DEDICATED_PORT.items():
        if key.lower() in facility_name:
            port = next((p for p in PORTS if p["name"] == port_name), None)
            if port:
                assigned_port = port
                dist = haversine_km(crow["LATITUD"], crow["LONGITUD"], port["lat"], port["lon"])
                break
    if not assigned_port:
        assigned_port, dist = nearest_port(crow["LATITUD"], crow["LONGITUD"], "concentrate")
    if assigned_port:
        downstream_edges.append({
            "FROM_NAME": crow["FACILITY_NAME"], "FROM_TYPE": "concentrator",
            "FROM_LAT": crow["LATITUD"], "FROM_LON": crow["LONGITUD"],
            "TO_NAME": assigned_port["name"], "TO_TYPE": "port",
            "TO_LAT": assigned_port["lat"], "TO_LON": assigned_port["lon"],
            "EDGE_TYPE": "concentrate_to_port", "PRODUCT_FORM": "concentrate",
            "OPERATOR": crow.get("OPERATOR_NAME", ""),
            "DISTANCE_KM": round(dist, 1) if isinstance(dist, (int, float)) else None,
        })

# D. SX-EW -> Port (Codelco consolidation first, then nearest cathode port)
for pidx, prow in inv[inv["CHAIN_STAGE"] == "sx_ew"].iterrows():
    if pd.isna(prow["LATITUD"]): continue
    facility_name = str(prow["FACILITY_NAME"]).lower()
    # Check Codelco cathode routing override
    assigned_port = None
    for key, port_name in CODELCO_CATHODE_ROUTING.items():
        if key.lower() in facility_name:
            port = next((p for p in PORTS if p["name"] == port_name), None)
            if port:
                assigned_port = port
                dist = haversine_km(prow["LATITUD"], prow["LONGITUD"], port["lat"], port["lon"])
                break
    # Also check dedicated port overrides
    if not assigned_port:
        for key, port_name in DEDICATED_PORT.items():
            if key.lower() in facility_name:
                port = next((p for p in PORTS if p["name"] == port_name), None)
                if port:
                    assigned_port = port
                    dist = haversine_km(prow["LATITUD"], prow["LONGITUD"], port["lat"], port["lon"])
                    break
    if not assigned_port:
        assigned_port, dist = nearest_port(prow["LATITUD"], prow["LONGITUD"], "cathode")
    if assigned_port:
        downstream_edges.append({
            "FROM_NAME": prow["FACILITY_NAME"], "FROM_TYPE": "sx_ew",
            "FROM_LAT": prow["LATITUD"], "FROM_LON": prow["LONGITUD"],
            "TO_NAME": assigned_port["name"], "TO_TYPE": "port",
            "TO_LAT": assigned_port["lat"], "TO_LON": assigned_port["lon"],
            "EDGE_TYPE": "sxew_to_port", "PRODUCT_FORM": "cathode_sxew",
            "OPERATOR": prow.get("OPERATOR_NAME", ""),
            "DISTANCE_KM": round(dist, 1),
        })

for etype in ["concentrate_to_smelter", "smelter_to_port", "concentrate_to_port", "sxew_to_port"]:
    n = sum(1 for e in downstream_edges if e["EDGE_TYPE"] == etype)
    print(f"  {etype:<25} {n:>4}")

# ── 3E. Unified edge table ──────────────────────────────────────────────

section_header("3E. UNIFIED SUPPLY CHAIN EDGES")

upstream = links[["MINE_NAME", "PLANT_NAME", "MINE_LAT", "MINE_LON",
                  "PLANT_LAT", "PLANT_LON", "SHARED_COMMODITIES",
                  "DISTANCE_KM", "PRODUCT_FORM"]].copy()
upstream.rename(columns={
    "MINE_NAME": "FROM_NAME", "PLANT_NAME": "TO_NAME",
    "MINE_LAT": "FROM_LAT", "MINE_LON": "FROM_LON",
    "PLANT_LAT": "TO_LAT", "PLANT_LON": "TO_LON",
    "SHARED_COMMODITIES": "COMMODITIES",
}, inplace=True)
upstream["FROM_TYPE"] = "mine"
upstream["TO_TYPE"] = "plant"
upstream["EDGE_TYPE"] = "mine_to_plant"

downstream_df = pd.DataFrame(downstream_edges)

# Enrich downstream commodities from inventory (instead of blanket "Copper")
def get_facility_commodities(facility_name):
    matches = inv[inv["FACILITY_NAME"] == facility_name]
    if len(matches) == 0:
        matches = inv[inv["FACILITY_NAME"].str.contains(facility_name[:12], case=False, na=False, regex=False)]
    if len(matches) == 0:
        return "Copper"
    comms = parse_comm_list(matches.iloc[0].get(comm_col, ""))
    if not comms: return "Copper"
    cu_prod = matches.iloc[0].get("COCHILCO_CU_2024_KMT", None)
    if (cu_prod is not None and pd.notna(cu_prod) and cu_prod > 0) or "Copper" in comms:
        return "Copper"
    return comms[0]

downstream_df["COMMODITIES"] = downstream_df["FROM_NAME"].apply(get_facility_commodities)
downstream_df["DISTANCE_KM"] = downstream_df["DISTANCE_KM"].round(1)

common_cols = ["FROM_NAME", "FROM_TYPE", "FROM_LAT", "FROM_LON",
               "TO_NAME", "TO_TYPE", "TO_LAT", "TO_LON",
               "EDGE_TYPE", "PRODUCT_FORM", "COMMODITIES", "DISTANCE_KM"]
for col in common_cols:
    for df in [upstream, downstream_df]:
        if col not in df.columns:
            df[col] = ""

edges = pd.concat([upstream[common_cols], downstream_df[common_cols]], ignore_index=True)

print(f"Unified edges: {len(edges)}")
for et, count in edges["EDGE_TYPE"].value_counts().items():
    print(f"  {et:<25} {count:>5}")



# ── Save state after Part 2 ─────────────────────────────────────────
_prev_path = os.path.join(DIR_PRELIM, "_pipeline_state_2.pkl")
with open(_prev_path, "rb") as _f:
    _save_state = pickle.load(_f)
_save_state["inv"] = inv
_save_state["links"] = links
_save_state["edges"] = edges
_save_state["common_cols"] = common_cols
_save_state["smelter_inv_map"] = smelter_inv_map
_out_path = os.path.join(DIR_PRELIM, "_pipeline_state_3.pkl")
with open(_out_path, "wb") as _f:
    pickle.dump(_save_state, _f)
print(f"State saved to {_out_path}")



3A. PROCESSING STAGE CLASSIFICATION
  extraction          187
  processing          129
  extraction_idle      80
  sx_ew                25
  concentration        22
  smelting             14
  refining              4

3B. SMELTER AND PORT SETUP
  Chuquicamata smelter           -> Chuquicamata SX-EW plant (oxide) and smelter
  Potrerillos smelter            -> Potrerillos SX-EW refinery and smelter
  Caletones smelter              -> El Teniente plant
  Altonorte smelter              -> Altonorte smelter
  Paipote smelter (H.V. Lira)    -> Hernán Videla Lira smelter
  Chagres smelter                -> Chagres smelter

  Ports saved: 13

3C. PRODUCT FORM + DOWNSTREAM EDGES
Product forms in mine-plant links:
PRODUCT_FORM
cathode_sxew    571
concentrate     440
blister          82
cathode_er       16

  Note: 342 links defaulted to 'concentrate' (no keyword match)
  concentrate_to_smelter      10
  smelter_to_port              9
  concentrate_to_port         12
  sxew_to_port            